In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
BASE_PATH = "/content/drive/MyDrive/multimodal_segmentation_qa"

DATA_PATH = f"{BASE_PATH}/data/unified"
MODEL_SAVE_PATH = f"{BASE_PATH}/model_colab.pth"

In [11]:
!pip install torch torchvision opencv-python tqdm

In [28]:
import os
import json
import random

images_dir = f"{DATA_PATH}/images"
masks_dir = f"{DATA_PATH}/masks"

image_files = os.listdir(images_dir)
mask_files = os.listdir(masks_dir)

mask_lookup = {}

for m in mask_files:
    key = m.replace("__segment_crack.png", "").replace("__segment_taping_area.png", "")
    mask_lookup[key] = m

data = []

for img in image_files:
    base = img.split(".rf")[0]

    if base in mask_lookup:
        mask_name = mask_lookup[base]

        prompt = "segment crack" if "crack" in img else "segment taping area"

        data.append({
            "image": f"{images_dir}/{img}",
            "mask": f"{masks_dir}/{mask_name}",
            "prompt": prompt
        })

print(f"Total matched pairs: {len(data)}")

crack = [x for x in data if x["prompt"] == "segment crack"]
taping = [x for x in data if x["prompt"] == "segment taping area"]

min_len = min(len(crack), len(taping))

random.shuffle(crack)
random.shuffle(taping)

balanced_data = crack[:min_len] + taping[:min_len]
random.shuffle(balanced_data)

print(f"Crack: {len(crack)}, Taping: {len(taping)}")
print(f"Balanced dataset size: {len(balanced_data)}")

with open(f"{DATA_PATH}/metadata_fixed.json", "w") as f:
    json.dump(balanced_data, f)

print("Final metadata saved!")

Total matched pairs: 6391
Crack: 5369, Taping: 1022
Balanced dataset size: 2044
Final metadata saved!


In [29]:
import json
import cv2
import torch
import os
from torch.utils.data import Dataset


class UnifiedDataset(Dataset):
    def __init__(self, metadata_path, img_size=256):
        with open(metadata_path, 'r') as f:
            data = json.load(f)

        self.img_size = img_size

        self.data = []
        for item in data:
            if os.path.exists(item["image"]) and os.path.exists(item["mask"]):
                self.data.append(item)

        print(f"Valid samples: {len(self.data)} / {len(data)}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        image = cv2.imread(item["image"])
        mask = cv2.imread(item["mask"], 0)

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        image = cv2.resize(image, (self.img_size, self.img_size))
        mask = cv2.resize(mask, (self.img_size, self.img_size))

        image = image / 255.0
        mask = mask / 255.0

        prompt_value = 1.0 if item["prompt"] == "segment crack" else 0.0

        h, w, _ = image.shape
        prompt_channel = torch.full((h, w, 1), prompt_value)

        image = torch.tensor(image, dtype=torch.float32)
        image = torch.cat([image, prompt_channel], dim=2)
        image = image.permute(2, 0, 1)

        mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)

        return image, mask

In [30]:
import torch.nn as nn
import torchvision.models as models

class BetterSegmentationModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = models.resnet18(weights="DEFAULT")
        self.encoder.conv1 = nn.Conv2d(4, 64, kernel_size=7, stride=2, padding=3, bias=False)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 2, stride=2),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 2, stride=2),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 2, stride=2),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 2, stride=2),
            nn.ReLU(),
            nn.Conv2d(32, 1, kernel_size=1)
        )

    def forward(self, x):
        x = self.encoder.conv1(x)
        x = self.encoder.bn1(x)
        x = self.encoder.relu(x)
        x = self.encoder.maxpool(x)

        x = self.encoder.layer1(x)
        x = self.encoder.layer2(x)
        x = self.encoder.layer3(x)
        x = self.encoder.layer4(x)

        x = self.decoder(x)
        return x

In [31]:
import torch
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import torch.nn.functional as F

BATCH_SIZE = 8
EPOCHS = 10
LR = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

dataset = UnifiedDataset(f"{DATA_PATH}/metadata_fixed.json", img_size=256)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=0)

model = BetterSegmentationModel().to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([8.0]).to(device))
optimizer = optim.Adam(model.parameters(), lr=LR)

def dice_score(pred, target):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.3).float()
    intersection = (pred * target).sum()
    return (2. * intersection) / (pred.sum() + target.sum() + 1e-6)

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for images, masks in loop:
        images, masks = images.to(device), masks.to(device)

        outputs = model(images)
        outputs = F.interpolate(outputs, size=masks.shape[-2:], mode='bilinear')

        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    model.eval()
    dice_total = 0

    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)

            outputs = model(images)
            outputs = F.interpolate(outputs, size=masks.shape[-2:], mode='bilinear')

            dice_total += dice_score(outputs, masks).item()

    print(f"\nEpoch {epoch+1}: Dice = {dice_total/len(val_loader):.4f}")

torch.save(model.state_dict(), MODEL_SAVE_PATH)
print("Model saved!")

Using: cuda
Valid samples: 2044 / 2044


Epoch 1/10: 100%|██████████| 205/205 [10:53<00:00,  3.19s/it, loss=1.09]



Epoch 1: Dice = 0.1767


Epoch 2/10: 100%|██████████| 205/205 [00:43<00:00,  4.71it/s, loss=1.22]



Epoch 2: Dice = 0.2976


...


Epoch 8: Dice = 0.4281


Epoch 9/10: 100%|██████████| 205/205 [00:43<00:00,  4.70it/s, loss=0.425]



Epoch 9: Dice = 0.4566


Epoch 10/10: 100%|██████████| 205/205 [00:42<00:00,  4.80it/s, loss=1.06]



Epoch 10: Dice = 0.4915
Model saved!


In [34]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import cv2
import os
from tqdm import tqdm
import numpy as np

MODEL_PATH = MODEL_SAVE_PATH
OUTPUT_DIR = f"{BASE_PATH}/outputs"
BATCH_SIZE = 4

os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

dataset = UnifiedDataset(f"{DATA_PATH}/metadata_fixed.json", img_size=256)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=0)

model = BetterSegmentationModel().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

def dice_score(pred, target):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.3).float()

    intersection = (pred * target).sum()
    return (2. * intersection) / (pred.sum() + target.sum() + 1e-6)


def iou_score(pred, target):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.3).float()

    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection

    return intersection / (union + 1e-6)

dice_total = 0
iou_total = 0

save_count = 0

with torch.no_grad():
    for i, (images, masks) in enumerate(tqdm(loader)):
        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)
        outputs = F.interpolate(outputs, size=masks.shape[-2:], mode='bilinear')

        dice_total += dice_score(outputs, masks).item()
        iou_total += iou_score(outputs, masks).item()
        preds = torch.sigmoid(outputs)
        preds = (preds > 0.3).float()

        for j in range(images.shape[0]):
            if save_count >= 12:
                break
            img = images[j][:3].permute(1, 2, 0).cpu().numpy()
            img = (img * 255).astype(np.uint8)

            gt = masks[j][0].cpu().numpy() * 255
            pred = preds[j][0].cpu().numpy() * 255

            combined = np.concatenate([
                img,
                np.stack([gt]*3, axis=-1).astype(np.uint8),
                np.stack([pred]*3, axis=-1).astype(np.uint8)
            ], axis=1)

            cv2.imwrite(f"{OUTPUT_DIR}/sample_{save_count}.png", combined)
            save_count += 1

avg_dice = dice_total / len(loader)
avg_iou = iou_total / len(loader)

print("\n===== FINAL RESULTS =====")
print(f"Mean Dice Score: {avg_dice:.4f}")
print(f"Mean IoU Score: {avg_iou:.4f}")
print(f"Saved {save_count} visualization images in {OUTPUT_DIR}")

Using: cuda
Valid samples: 2044 / 2044


100%|██████████| 511/511 [00:43<00:00, 11.71it/s]


===== FINAL RESULTS =====
Mean Dice Score: 0.5914
Mean IoU Score: 0.4421
Saved 12 visualization images in /content/drive/MyDrive/multimodal_segmentation_qa/outputs
